### Paper History: TD(λ) and Convergence Theory

| Year | Paper | Main Contribution | Historical Note |
|---|---|---|---|
| 1988 | Sutton, *Learning to Predict by the Methods of Temporal Differences* | Introduced the **TD(λ) family**, including TD(0) and TD(1). Proved convergence in expectation for linear TD(0) under specific absorbing Markov process assumptions. | TD(λ) was already invented. Convergence for intermediate values $0<\lambda<1$ remained an open problem. |
| 1992 | Dayan, *The Convergence of TD(λ) for General λ* | Extended Sutton's convergence-in-expectation result to general TD(λ). Also discussed almost-sure convergence for a special modified case. | Resolved the main theoretical question left open by Sutton for general λ. |
| 1994 | Dayan & Sejnowski, *TD(λ) Converges with Probability 1* | Established almost-sure convergence for a suitably modified form of linear TD(λ), and analyzed convergence rates. | Strengthened the convergence guarantees beyond convergence in expectation. |
| 1997 | Tsitsiklis & Van Roy, *An Analysis of Temporal-Difference Learning with Function Approximation* | Proved almost-sure convergence of online linear TD(λ) under specified Markov-chain assumptions, characterized its limit, and bounded approximation error. | Clarified why linear function approximation can converge while certain nonlinear or improperly sampled variants can diverge. |

### Notes

1. **Invention vs. Proof:** Sutton introduced TD(λ) in 1988. Dayan did not invent TD(λ); he extended its convergence theory.

2. **TD(n) vs. TD(λ):**
   - **n-step TD:** Uses a fixed number of transitions before bootstrapping.
   - **TD(λ):** Uses eligibility traces; its forward view combines multiple n-step returns through λ-dependent weighting.
   - **Important:** TD(0) corresponds to one-step TD, whereas TD(1) is related to Monte Carlo learning in episodic settings.

3. - Convergence in expectation: The average prediction across independent training runs approaches the target value, even if individual runs do not converge.
   - Almost-sure convergence: Individual training runs converge to the target value with probability 1, allowing exceptions only on a probability-zero set.

---

### convergence summary

| Setting                                         | Convergence Guarantee                                                                                  |
| ----------------------------------------------- | ------------------------------------------------------------------------------------------------------ |
| Exact Bellman n-step Operator, \\(\gamma < 1\\) | Yes, converges to \\(V^\pi\\)                                                                          |
| Tabular, Fixed Policy, On-policy n-step TD      | Yes, under standard sampling and learning-rate conditions                                              |
| On-policy Linear TD(λ)                          | Yes, under appropriate assumptions; may converge to an approximation rather than the exact \\(V^\pi\\) |
| Off-policy + Function Approximation             | Not universally guaranteed                                                                             |
| Nonlinear Neural-Network TD                     | Not universally guaranteed                                                                             |

### Why Mix n-Step Returns?

From first principles, an n-step return combines two sources of information:

$$
G_t^{(n)}
=
\underbrace{\sum_{k=0}^{n-1}\gamma^k R_{t+k+1}}_{\text{Observed rewards}}
+
\underbrace{\gamma^n\hat V(S_{t+n})}_{\text{Estimated future}}
$$

Both sources are imperfect:

- **Observed rewards:** These are actual observations, but they come from a single random trajectory and therefore introduce **sampling variance**.
- **Estimated future:** This comes from the critic, allowing us to estimate future returns without sampling the entire trajectory. However, the critic may introduce **systematic prediction errors (bias)**.

As \(n\) increases, we rely more on observed rewards and less on the critic's estimates.

This leads to a natural question:

> If the critic can be inaccurate and sampled trajectories are inherently random, why force ourselves to choose a single fixed \(n\)?

Instead, we can construct a weighted mixture of n-step returns:

$$
G_t^{\text{mix}} = \sum_n w_n G_t^{(n)}
$$

where:

$$
w_n \geq 0, \qquad \sum_n w_n = 1
$$

In other words, **each n-step estimator contributes to the final estimate**, rather than relying on a single horizon.

This gives us a way to balance the trade-off between sampling variance and potential bootstrapping bias.


---

### Intuition Behind the Design of λ-Return

Suppose we want to satisfy three design requirements:

1. **No fixed horizon:** We do not need to decide in advance how many steps to observe.
2. **Single parameter:** We only need one parameter to control our reliance on bootstrapping.
3. **Consistent rule:** We can apply the same decision rule at every step.

A fixed continuation probability \(\lambda\) is a natural choice that satisfies all three requirements.

An additional benefit is that **λ-return can be expressed recursively**:

$$
G_t^\lambda
=
r_{t+1}+\gamma
\left[
(1-\lambda)V(s_{t+1})
+\lambda G_{t+1}^\lambda
\right]
$$

*Assuming the transition is nonterminal; at termination, the continuation value is zero.*

The intuition is:

> First, I observe one step of actual reward. Then, for the remaining future return, I assign weight \(1-\lambda\) to the Critic's prediction and weight \(\lambda\) to continuing forward and observing more actual rewards.

### The Role of λ: Bias–Variance Tradeoff

When the Critic is imperfect, **λ controls the tradeoff between bootstrapping error and sampling variance**.

- **Smaller λ:** Relies more heavily on the Critic, typically reducing sampling variance but potentially introducing more bootstrapping bias.
- **Larger λ:** Relies more heavily on observed rewards, reducing dependence on potentially inaccurate value estimates but typically increasing sampling variance.